In [1]:
import pandas as pd 
import numpy as np 


In [2]:
df=pd.read_csv("Dataset_insuficiencia_renal_crnica.csv")
df.columns

Index(['Identificación del paciente', 'Edad', 'Presión arterial',
       'Gravedad específica de la orina', 'Albúmina en orina',
       'Azúcar en orina', 'Glóbulos rojos', 'Células de pus',
       'Grupos de células de pus', 'Bacterias', 'Glucosa en sangre aleatoria',
       'Urea en sangre', 'Creatinina sérica', 'Sodio en sangre',
       'Potasio en sangre', 'Hemoglobina', 'Volumen corpuscular empaquetado',
       'Recuento de glóbulos blancos', 'Recuento de glóbulos rojos', 'Apetito',
       'Insuficiencia renal crónica'],
      dtype='object')

In [3]:
columns = [
    "Patient Identification",
    "Age",
    "Blood Pressure",
    "Urine Specific Gravity",
    "Urine Albumin",
    "Urine Sugar",
    "Red Blood Cells",
    "Pus Cells",
    "Pus Cell Clumps",
    "Bacteria",
    "Random Blood Glucose",
    "Blood Urea",
    "Serum Creatinine",
    "Blood Sodium",
    "Blood Potassium",
    "Hemoglobin",
    "Packed Cell Volume",
    "White Blood Cell Count",
    "Red Blood Cell Count",
    "Appetite",
    "Diagnosis"
]


In [4]:
df.columns = columns
df.head()

,Patient Identification,Age,Blood Pressure,Urine Specific Gravity,Urine Albumin,Urine Sugar,Red Blood Cells,Pus Cells,Pus Cell Clumps,Bacteria,...,Blood Urea,Serum Creatinine,Blood Sodium,Blood Potassium,Hemoglobin,Packed Cell Volume,White Blood Cell Count,Red Blood Cell Count,Appetite,Diagnosis
0,0,48,178,1010.0,2.0,2,anormal,anormal,presente,presente,...,195.0,9.08,124.0,6.0,8.0,25.0,11319,3.8,bajo,si
1,1,7,138,1014.0,1.0,4,anormal,anormal,presente,presente,...,111.0,7.88,133.0,6.8,11.0,27.0,11018,2.9,bajo,si
2,2,62,165,1014.0,2.0,5,anormal,anormal,presente,presente,...,149.0,12.89,130.0,6.0,8.0,27.0,5479,4.4,bajo,si
3,3,48,141,1007.0,4.0,4,anormal,anormal,presente,presente,...,98.0,5.02,133.0,5.8,11.0,35.0,7817,3.2,bajo,si
4,4,51,175,1015.0,4.0,5,anormal,anormal,presente,presente,...,152.0,7.32,132.0,6.8,9.0,30.0,3055,3.9,bajo,si


In [5]:
df = df.replace({
    "presente": "present",
    "ausente": "absent",
    "anormal": "abnormal",
    "bueno": "good",
    "bajo": "bad",
    "si": "yes",
    "no": "no"
})


In [6]:
df= df.dropna()

In [7]:

df.shape

(15024, 21)

In [8]:
import pandas as pd

# Select only numeric columns for outlier detection
numeric_cols = df.select_dtypes(include=['float64','int64']).columns

# Function to remove outliers using IQR
def remove_outliers_iqr(data, cols):
    for col in cols:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Keep only values within bounds
        data = data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]
    return data

# Apply outlier removal
df_clean = remove_outliers_iqr(df, numeric_cols)

print("Original shape:", df.shape)
print("After outlier removal:", df_clean.shape)


Original shape: (15024, 21)
After outlier removal: (10987, 21)


In [9]:
df_clean.drop("Patient Identification",axis=1).to_csv("cleaned_Data.csv")

In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Copy dataset
df_encoded = df_clean.copy()

# Initialize LabelEncoder
le = LabelEncoder()

# Apply label encoding to all object (categorical) columns
for col in df_encoded.select_dtypes(include=['object']).columns:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# Separate features and target
X = df_encoded.drop(["Diagnosis","Patient Identification"], axis=1)
y = df_encoded["Diagnosis"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Random Forest Classifier
rf = RandomForestClassifier()

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.9990900818926297

Confusion Matrix:
 [[1028    1]
 [   1 1168]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1029
           1       1.00      1.00      1.00      1169

    accuracy                           1.00      2198
   macro avg       1.00      1.00      1.00      2198
weighted avg       1.00      1.00      1.00      2198



In [11]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Copy dataset
df_encoded = df_clean.copy()

# Initialize LabelEncoder
le = LabelEncoder()

# Apply label encoding to all object (categorical) columns
for col in df_encoded.select_dtypes(include=['object']).columns:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# Separate features and target
X = df_encoded.drop(["Diagnosis","Patient Identification"], axis=1)
y = df_encoded["Diagnosis"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Random Forest Classifier
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.9990900818926297

Confusion Matrix:
 [[1028    1]
 [   1 1168]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1029
           1       1.00      1.00      1.00      1169

    accuracy                           1.00      2198
   macro avg       1.00      1.00      1.00      2198
weighted avg       1.00      1.00      1.00      2198



In [12]:
report = classification_report(y_test, y_pred, output_dict=True)
for label, metrics in report.items():
    if isinstance(metrics, dict):
        print(f"{label}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1-score']:.4f}")


0: Precision=0.9990, Recall=0.9990, F1=0.9990
1: Precision=0.9991, Recall=0.9991, F1=0.9991
macro avg: Precision=0.9991, Recall=0.9991, F1=0.9991
weighted avg: Precision=0.9991, Recall=0.9991, F1=0.9991
